# DARF v4 — Face-Orientation Lock

**Sorun teşhisi:** v1–v3'teki Daenerys yan-profile sorunu artık identity sorunu değil; **LoRA training-data orientation bias'ı**. Synthetic OpenPose iskelet yüzün kameraya dönük olduğunu yeterince güçlü söylemiyor → Daenerys LoRA `dae woman` embedding'i gevşek bir orientation hint'i ile karşılaşınca kendi bias'ına düşüyor (yan profil).

v4 stratejisi:
1. **Daenerys α'yı düşür** — çok yüksek α identity verir ama yön bias'ı baskınlaşır. Mid=0.75, up=1.00.
2. **Stage 2'yi minimize et** — `refine_strength=0.25–0.30`, `identity_ctrl_scale=1.0`. Stage 2 yüzü yeniden çizmek yerine sadece *cilalama* yapar.
3. **`front-facing` token'ını Daenerys region'ına bağla** — visual_description ve attention_phrases'e `front-facing woman looking directly at the camera, symmetrical face` ekle.
4. **Face-local refine geniş crop + düşük strength** — `crop_pad_ratio=1.0` (saç + omuzları dahil), `refine_strength=0.28`. Yüksek strength yüzü tekrar yan profile çekmesin.
5. **Negative prompt'a explicit orientation cezası** — "side profile, turned head, looking sideways".

**Yeni bir base model değiştirmiyoruz, LoRA'ları değiştirmiyoruz** — sadece inference-time control.

## Bölüm 1 — Setup

In [ ]:
import os
REPO_URL = "https://github.com/melik3kara/LoRA-scale-adaptive-feedback.git"
REPO_DIR = "/workspace/LoRA-scale-adaptive-feedback"
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Pulling latest...")
    !cd {REPO_DIR} && git pull

In [ ]:
!pip install -q gdown
os.makedirs(f"{REPO_DIR}/data/loras", exist_ok=True)
DRIVE_FOLDER_ID = "1L6NA_AT5HGSNOcKjw7gECbPKnaN86Ix5"
import glob
if not glob.glob(f"{REPO_DIR}/data/loras/*.safetensors"):
    !gdown --folder "https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}" -O {REPO_DIR}/data/loras/
    import shutil
    for f in glob.glob(f"{REPO_DIR}/data/loras/**/*.safetensors", recursive=True):
        target = os.path.join(f"{REPO_DIR}/data/loras", os.path.basename(f))
        if f != target:
            shutil.move(f, target)

In [ ]:
os.environ["HF_HOME"] = "/workspace/cache/huggingface"
os.environ["INSIGHTFACE_HOME"] = "/workspace/cache/insightface"
os.makedirs("/workspace/cache/huggingface", exist_ok=True)
os.makedirs("/workspace/cache/insightface", exist_ok=True)

!pip install -q diffusers==0.30.3 transformers==4.44.2 peft controlnet-aux insightface onnxruntime-gpu
!pip install -q "pydantic<2.10" "typing_extensions>=4.13.0"
!pip install -q "mediapipe==0.10.14" "protobuf<5" scipy

In [ ]:
os.chdir(REPO_DIR)
import sys
if "src" not in sys.path:
    sys.path.insert(0, "src")
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}  VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")

## Bölüm 2 — Pipeline + Scorers + Evaluator

In [ ]:
from pipeline import load_identities, build_pipeline
from scorer import FaceScorer
from pose_scorer import PoseScorer
from attribute_scorer import AttributeScorer
from evaluator import Evaluator
from PIL import Image
from IPython.display import display

identities = load_identities()

# === ORIENTATION LOCK: identities'i front-facing emphasis ile override et ===
# identities.yaml'a dokunmadan, bu run boyunca metadata'yı zenginleştir.
FRONT_FACE_DAENERYS = (
    "front-facing woman looking directly at the camera, symmetrical face, "
    "long straight platinum silver-white hair, violet eyes, pale skin, "
    "wearing a regal blue dress"
)
FRONT_FACE_HERMIONE = (
    "front-facing young woman looking directly at the camera, symmetrical face, "
    "long bushy brown hair, hazel eyes, fair skin, wearing a black Hogwarts school robe"
)

identities["hermione"]["visual_description"] = FRONT_FACE_HERMIONE
identities["daenerys"]["visual_description"] = FRONT_FACE_DAENERYS

# Attention phrases — front-facing token'ı bu region'a bağlanır
identities["hermione"]["attention_phrases"] = [
    "Hermione_Granger",
    "front-facing young woman looking directly at the camera",
    "long bushy brown hair",
    "black Hogwarts school robe",
]
identities["daenerys"]["attention_phrases"] = [
    "dae woman",
    "front-facing woman looking directly at the camera",
    "long straight platinum silver-white hair",
    "regal blue dress",
]

# Negative features — orientation cezası eklendi
identities["hermione"]["negative_features"] = (
    "blonde hair, silver hair, platinum hair, white hair, crown, "
    "side profile, turned head, looking sideways, profile view"
)
identities["daenerys"]["negative_features"] = (
    "brown hair, brunette, bushy hair, curly hair, witch robe, school uniform, "
    "side profile, turned head, looking sideways, profile view"
)

print("Identities augmented for front-facing orientation lock.")
for k in identities:
    print(f"  {k}: {identities[k]['visual_description'][:80]}...")

In [ ]:
# Global negative prompt'a da orientation cezası ekle
import pipeline as P
P._BASE_NEGATIVE = (
    "blurry, low quality, deformed face, three women, three people, "
    "third person, third woman, extra person, another woman, "
    "woman in background, person behind, distant person, crowd, duplicate, "
    "side profile, profile view, turned head, looking sideways, looking away, "
    "three-quarter view, back view"
)
print("_BASE_NEGATIVE augmented with orientation penalty")

In [ ]:
pipe = build_pipeline(identities)

face_scorer = FaceScorer(reference_dir="data/reference_faces", device="cuda")
pose_scorer = PoseScorer()
attr_scorer = AttributeScorer(identities, device="cuda")
evaluator   = Evaluator(device="cuda")

ref_embeds = {k: evaluator.extract_face_embedding(
    Image.open(identities[k]["reference_image"]).convert("RGB")) for k in identities}

names = list(identities.keys())
identity_regions = {names[0]: (0,0,512,1024), names[1]: (512,0,1024,1024)}

# Helper
import numpy as np, cv2
def detected_face_count(img):
    img_bgr = cv2.cvtColor(np.array(img.convert("RGB")), cv2.COLOR_RGB2BGR)
    return len(evaluator.face_app.get(img_bgr))

def show_eval(label, image):
    sim_h, sim_d = evaluator.detect_and_assign_faces(
        image, ref_embeds["hermione"], ref_embeds["daenerys"]
    )
    n = detected_face_count(image)
    print(f"[Eval/{label}] arc_h={sim_h:+.3f}  arc_d={sim_d:+.3f}  faces={n}")
    return {"label": label, "sim_h": sim_h, "sim_d": sim_d, "faces": n}

print("\nReady.")

## Bölüm 3 — Face-Aware Dense Pose

In [ ]:
from face_aware_pose import make_face_aware_pose, get_face_aware_target_keypoints

pose_dense = make_face_aware_pose(1024, 1024, front_facing=True, dense_face=True)
target_keypoints = get_face_aware_target_keypoints(front_facing=True)

print("Dense face-aware pose:")
display(pose_dense.resize((384, 384)))

## Bölüm 4 — v4 Ana Konfigürasyon

GPT'nin önerdiği parametreler + bizim mekanizmalarımız:

- Daenerys α: **`{down: 0.30, mid: 0.75, up: 1.00}`** (v3'teki 1.50'den çok düşük)
- `identity_ctrl_scale=1.00` (pose lock max)
- `refine_strength=0.25` (v3'teki 0.40'tan da düşük)
- BG lock Stage 1'de aktif
- Dense face pose

In [ ]:
from pipeline import generate_two_stage

v4_config = dict(
    layout_lora_scales={"hermione": 0.10, "daenerys": 0.25},
    identity_lora_scales={
        "hermione": {"down": 0.10, "mid": 0.20, "up": 0.35},
        "daenerys": {"down": 0.30, "mid": 0.75, "up": 1.00},   # ← çok daha hafif
    },
    layout_ctrl_scale=1.00,
    identity_ctrl_scale=1.00,                                     # ← max pose lock
    refine_strength=0.25,                                          # ← min refine
    refine_steps=28, layout_steps=28,
    use_regional_attention=True,
    use_spatial_lora_gate=True,
    spatial_gate_block_floor={"down": 0.15, "mid": 0.08, "up": 0.00},
    use_bg_lock=True,
    bg_lock_ratio=0.40,
    bg_lock_padding=24,
    bg_lock_in_layout=True,
    bg_lock_in_identity=False,
    identity_regions=identity_regions,
)

stage1, stage2 = generate_two_stage(
    pipe, identities, pose_dense,
    seed=42,
    **v4_config,
)

print("\n[Stage 1 — layout, BG-locked, minimal LoRA]");  display(stage1.resize((512,512)))
print("[Stage 2 — minimal-strength identity refine]");    display(stage2.resize((512,512)))
show_eval("v4_main_stage1", stage1)
show_eval("v4_main_stage2", stage2)

## Bölüm 5 — Face-Local Refine (geniş crop, düşük strength)

Yüksek strength = yüzü yeniden çizer = orientation tekrar bozulur.

Düşük strength + büyük crop (saç+omuzlar dahil) = sadece detail cilası, orientation korunur.

In [ ]:
from pipeline import face_local_refine

v4_final = face_local_refine(
    pipe, identities, stage2,
    face_scorer, identity_regions,
    refine_strength=0.28,        # ← düşük: yüzü yeniden yan profile çekmesin
    crop_pad_ratio=1.0,          # ← geniş: saç + omuzlar dahil
    refine_steps=25,
    seed=42,
    feather=32,                  # ← daha yumuşak edge
)

print("\n[v4 final — face-local refined]")
display(v4_final.resize((512, 512)))
show_eval("v4_final", v4_final)

## Bölüm 6 — Daenerys α Sweep (orientation-bias eğrisini ölç)

Akademik olarak kıymetli ablation: Daenerys up-block α'sı yükseldikçe identity güçlenir ama orientation drift artar. Sweet spot'u eğride göster.

In [ ]:
alpha_sweep = [
    ("d_up_0.6",  {"down": 0.20, "mid": 0.45, "up": 0.60}),
    ("d_up_0.8",  {"down": 0.25, "mid": 0.60, "up": 0.80}),
    ("d_up_1.0",  {"down": 0.30, "mid": 0.75, "up": 1.00}),
    ("d_up_1.2",  {"down": 0.35, "mid": 0.90, "up": 1.20}),
    ("d_up_1.5",  {"down": 0.40, "mid": 1.10, "up": 1.50}),
]

alpha_results = {}
for label, dscale in alpha_sweep:
    print(f"\n=== {label} ===")
    cfg = {**v4_config}
    cfg["identity_lora_scales"] = {
        "hermione": {"down": 0.10, "mid": 0.20, "up": 0.35},
        "daenerys": dscale,
    }
    _, s2 = generate_two_stage(pipe, identities, pose_dense, seed=42, **cfg)
    alpha_results[label] = s2
    display(s2.resize((384, 384)))
    show_eval(label, s2)

## Bölüm 7 — Multi-Seed (rapor için final sayılar)

In [ ]:
seeds = [42, 123, 777]
v4_seed_imgs = {}
v4_seed_finals = {}

for s in seeds:
    print(f"\n=== v4 seed {s} ===")
    _, s2 = generate_two_stage(pipe, identities, pose_dense, seed=s, **v4_config)
    v4_seed_imgs[s] = s2
    final = face_local_refine(
        pipe, identities, s2, face_scorer, identity_regions,
        refine_strength=0.28, crop_pad_ratio=1.0,
        refine_steps=25, seed=s, feather=32,
    )
    v4_seed_finals[s] = final
    print("Stage 2:");    display(s2.resize((384, 384)))
    print("Final:");      display(final.resize((384, 384)))
    show_eval(f"v4_seed{s}_stage2", s2)
    show_eval(f"v4_seed{s}_final",  final)

## Bölüm 8 — Ablation Table + ZIP

In [ ]:
import pandas as pd

def collect(label, image):
    fc = face_scorer.score_image(image, identity_regions)
    pc = pose_scorer.score_image(image, target_keypoints, identity_regions)
    ac = attr_scorer.score_image(image, identity_regions)
    sim_h, sim_d = evaluator.detect_and_assign_faces(
        image, ref_embeds["hermione"], ref_embeds["daenerys"]
    )
    n = detected_face_count(image)
    rows = []
    for k, eval_sim in zip(["hermione", "daenerys"], [sim_h, sim_d]):
        rows.append({
            "method":         label,
            "identity":       k,
            "arcface_self":   round(fc[k]["arcface"], 4),
            "dominance":      round(fc[k]["dominance"], 4),
            "pose":           round(pc[k]["pose"], 4),
            "attr_margin":    round(ac[k]["margin"], 4),
            "eval_arcface":   round(eval_sim, 4),
            "face_count":     n,
        })
    return rows

rows = []
if "stage2"   in dir(): rows += collect("v4_main_stage2", stage2)
if "v4_final" in dir(): rows += collect("v4_final",       v4_final)
if "alpha_results" in dir():
    for label, img in alpha_results.items():
        rows += collect(label, img)
if "v4_seed_imgs" in dir():
    for s, img in v4_seed_imgs.items():
        rows += collect(f"v4_seed{s}_stage2", img)
if "v4_seed_finals" in dir():
    for s, img in v4_seed_finals.items():
        rows += collect(f"v4_seed{s}_final", img)

df = pd.DataFrame(rows)
out = "data/results/darf_v4"
os.makedirs(out, exist_ok=True)
df.to_csv(f"{out}/ablation.csv", index=False)
df

In [ ]:
# Save + ZIP
if "stage1"   in dir(): stage1.save(f"{out}/v4_main_stage1.png")
if "stage2"   in dir(): stage2.save(f"{out}/v4_main_stage2.png")
if "v4_final" in dir(): v4_final.save(f"{out}/v4_final.png")
if "alpha_results" in dir():
    for label, img in alpha_results.items():
        img.save(f"{out}/{label}.png")
if "v4_seed_imgs" in dir():
    for s, img in v4_seed_imgs.items():
        img.save(f"{out}/v4_seed{s}_stage2.png")
if "v4_seed_finals" in dir():
    for s, img in v4_seed_finals.items():
        img.save(f"{out}/v4_seed{s}_final.png")
pose_dense.save(f"{out}/pose_skeleton.png")

import shutil
from IPython.display import FileLink
shutil.make_archive("/workspace/darf_v4_results", "zip", out)
print("Zip ready: /workspace/darf_v4_results.zip")
FileLink("/workspace/darf_v4_results.zip")

## Akademik Çerçeve

v4 reframes the persistent Daenerys side-profile failure mode as a **LoRA training-data orientation bias** problem rather than an identity-strength problem.

Key observation: in v1–v3 we increased Daenerys α to recover identity, but at high α the LoRA's training-time pose distribution (predominantly side-profile reference shots) overrides ControlNet's pose conditioning. v4 inverts the trade-off:

1. **Lower asymmetric α** — Daenerys mid=0.75, up=1.00 (down from 1.20/1.50 in v3). Identity is partially recovered by the spatial gate's per-block isolation, not by raw scale.
2. **Maximum ControlNet at refinement** — `identity_ctrl_scale=1.0` ensures Stage 2 cannot override pose orientation.
3. **Minimal Stage 2 strength** — `refine_strength=0.25` means Stage 2 contributes only ~25% of the denoising trajectory; the layout/orientation locked in Stage 1 is preserved.
4. **Wide-crop, low-strength face refinement** — `crop_pad_ratio=1.0, refine_strength=0.28` preserves the orientation set in Stage 1/2 while restoring per-identity face detail.
5. **Front-facing token binding** — "front-facing woman looking directly at the camera" injected into both `visual_description` and `attention_phrases`, and "side profile / turned head" added to per-identity `negative_features`. The regional attention mask binds these tokens to their identity's spatial region.

Final paper claim:
> *"DARF v4 demonstrates that LoRA orientation bias is largely independent of identity strength: by decoupling these two axes via low-strength asymmetric scaling and Stage-1 pose locking, we restore frontal pose without sacrificing identity, where naive scale increases trade them off."*